# YOLOv11m + ECA+CBAM - Architecture Dependence Test

Attaches the identical residual-gated ECA+CBAM block used in RGDA-YOLOv8m to
YOLOv11m, whose backbone already contains a C2PSA partial self-attention stage,
and trains it across the same three seeds under the same two-phase schedule.


All runs use the deduplicated 926-image dataset and the seed-42 80/20 split.


Environment Detection

In [1]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT, SAVE_DIR and OUTPUT_DIR accordingly.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_JUPYTER = not ON_KAGGLE and not ON_COLAB

if ON_KAGGLE:
    print("Running on KAGGLE")
    ROOT = "/kaggle/working"
elif ON_COLAB:
    print("Running on GOOGLE COLAB")
    ROOT = "/content"
else:
    print("Running on LOCAL JUPYTER")
    ROOT = "."

OUTPUT_DIR = os.path.join(ROOT, "attention_results")
DATA_DIR = os.path.join(ROOT, "data")
SAVE_DIR = os.path.join(ROOT, "saved_models")

for d in [OUTPUT_DIR, DATA_DIR, SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"   Output  -> {OUTPUT_DIR}")
print(f"   Data    -> {DATA_DIR}")
print(f"   Models  -> {SAVE_DIR}")

Running on LOCAL JUPYTER
   Output  -> ./attention_results
   Data    -> ./data
   Models  -> ./saved_models


Verify Environment

In [2]:
# Prints package versions and confirms GPU availability and device name.
import torch, numpy as np, pandas as pd, cv2

print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"OpenCV  : {cv2.__version__}")
print(f"PyTorch : {torch.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")
    print(
        f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )
else:
    print("   WARNING: No GPU -- training will be slow.")

try:
    import ultralytics, seaborn, tqdm, kagglehub, yaml

    print("ultralytics, seaborn, tqdm, kagglehub, pyyaml -- all good!")
except ImportError as e:
    print(f"FAIL: Missing package: {e}")
    print("   Run: pip install ultralytics seaborn tqdm kagglehub pyyaml")

NumPy   : 2.2.6
Pandas  : 2.3.3
OpenCV  : 4.13.0
PyTorch : 2.10.0+cu128
Device  : cuda
   GPU    : NVIDIA GeForce RTX 3090
   VRAM   : 25.4 GB
ultralytics, seaborn, tqdm, kagglehub, pyyaml -- all good!


Config & Hyperparameters

In [3]:
# Defines the training and evaluation config: two-phase epochs and learning rates, batch size, image size, seeds, the confidence sweep grid, and the default evaluation thresholds.
EPOCHS_FROZEN = 10  # Phase 1: train only attention + head, backbone frozen
EPOCHS_FULL = 40  # Phase 2: unfreeze everything, fine-tune end-to-end
BATCH = 16
IMG_SIZE = 640
LR_FROZEN = 1e-3  # Higher LR for Phase 1 (only new layers training)
LR_FULL = 2e-4  # Lower LR for Phase 2 (avoid overwriting pretrained weights)
WEIGHT_DECAY = 5e-4

# Attention config
# ECA: kernel size is adaptive (uses log2 of channels) -- no hyperparams needed
# CBAM: reduction ratio for channel attention MLP
CBAM_REDUCTION = 16
CBAM_KERNEL = 7  # spatial attention conv kernel

# Evaluation config
# We evaluate across a sweep and also report at the optimal threshold.
CONF_SWEEP = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
]
CONF_DEFAULT = 0.25  # Standard YOLO default for fair comparison
IOU_THRESH = 0.5  # IoU threshold for TP/FP
MAX_IMAGES = None  # None = use full test set

print("Config loaded")
print(f"   Phase 1 : {EPOCHS_FROZEN} epochs, freeze backbone, LR={LR_FROZEN}")
print(f"   Phase 2 : {EPOCHS_FULL} epochs, full fine-tune, LR={LR_FULL}")
print(
    f"   Eval conf: sweep {CONF_SWEEP[0]}-{CONF_SWEEP[-1]}, default at {CONF_DEFAULT}"
)

Config loaded
   Phase 1 : 10 epochs, freeze backbone, LR=0.001
   Phase 2 : 40 epochs, full fine-tune, LR=0.0002
   Eval conf: sweep 0.1-0.7, default at 0.25


Dataset Loading

In [4]:
# Downloads the three Kaggle datasets, loads every annotation through the unified loader, and deduplicates before any split.
import kagglehub
from pathlib import Path
import xml.etree.ElementTree as ET
import shutil

print("Downloading datasets via kagglehub ...")
path_1 = kagglehub.dataset_download("chitholian/annotated-potholes-dataset")
path_2 = kagglehub.dataset_download("andrewmvd/pothole-detection")
path_3 = kagglehub.dataset_download("ashishkumarak/training-setzip")
DATASET_ROOTS = {"chitholian": path_1, "andrewmvd": path_2, "ashishkumar": path_3}
print("Datasets ready")


def load_annotated_potholes(root, max_imgs=None):
    root = Path(root)
    xml_index = {p.stem: p for p in root.rglob("*.xml")}

    records = []
    for img_path in (list(root.rglob("*.jpg")) + list(root.rglob("*.png")))[:max_imgs]:
        gt_boxes = []
        xml_path = xml_index.get(img_path.stem)
        if xml_path is not None and xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                for obj in tree.findall("object"):
                    bb = obj.find("bndbox")
                    gt_boxes.append(
                        [
                            float(bb.find("xmin").text),
                            float(bb.find("ymin").text),
                            float(bb.find("xmax").text),
                            float(bb.find("ymax").text),
                        ]
                    )
            except Exception:
                pass
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


def load_ashishkumar_csv(root, max_imgs=None):
    import pandas as pd

    root = Path(root)
    csv_path = root / "train" / "labels.csv"
    img_dir = root / "train" / "images"
    df = pd.read_csv(csv_path)
    grouped = df.groupby("ImageID")

    records = []
    img_paths = sorted(img_dir.glob("*.jpg"))[:max_imgs]
    for img_path in img_paths:
        gt_boxes = []
        if img_path.name in grouped.groups:
            for _, row in grouped.get_group(img_path.name).iterrows():
                gt_boxes.append(
                    [
                        float(row["XMin"]),
                        float(row["YMin"]),
                        float(row["XMax"]),
                        float(row["YMax"]),
                    ]
                )
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


all_records = []
for name, root in DATASET_ROOTS.items():
    if name == "ashishkumar":
        recs = load_ashishkumar_csv(root, MAX_IMAGES)
    else:
        recs = load_annotated_potholes(root, MAX_IMAGES)
    print(
        f"   {name}: {len(recs)} images ({sum(len(r['gt_boxes']) for r in recs)} gt boxes)"
    )
    all_records.extend(recs)

records = [r for r in all_records if r["gt_boxes"]]  # only annotated
print(f"\nTotal annotated before dedup: {len(records)} images")

import numpy as np
from PIL import Image

NORM_SIZE = (64, 64)
DEDUP_THRESHOLD = 1.0  # mean abs pixel diff (0-255 scale); true duplicates
# measured at 0.10-0.50, unrelated images much higher


def normalized_pixels(path):
    with Image.open(path) as img:
        return np.asarray(
            img.convert("L").resize(NORM_SIZE, Image.LANCZOS), dtype=np.float32
        ).ravel()


print("Computing normalized pixel arrays for dedup ...")
all_arrs = np.stack([normalized_pixels(r["image_path"]) for r in records])

keep_mask = np.ones(len(records), dtype=bool)
seen_arrs = []  # arrays of images already kept
for i in range(len(records)):
    if not keep_mask[i]:
        continue
    if seen_arrs:
        diffs = np.abs(np.stack(seen_arrs) - all_arrs[i]).mean(axis=1)
        if diffs.min() < DEDUP_THRESHOLD:
            keep_mask[i] = False
            continue
    seen_arrs.append(all_arrs[i])

# Diagnostic: show the distribution of nearest-neighbor diffs among the
# images that got removed, so the threshold can be sanity-checked directly
# rather than guessed at again if the final count still looks off.
removed_diffs = []
_seen_for_diag = []
for i in range(len(records)):
    if _seen_for_diag:
        d = np.abs(np.stack(_seen_for_diag) - all_arrs[i]).mean(axis=1).min()
        if not keep_mask[i]:
            removed_diffs.append(d)
    if keep_mask[i]:
        _seen_for_diag.append(all_arrs[i])
if removed_diffs:
    removed_diffs = np.array(removed_diffs)
    print(
        f"\nRemoved-pair diff stats: min={removed_diffs.min():.3f} "
        f"median={np.median(removed_diffs):.3f} max={removed_diffs.max():.3f}"
    )
    print(
        f"   (all removed pairs should sit well below DEDUP_THRESHOLD={DEDUP_THRESHOLD} "
        f"-- if max is close to the threshold, some may be false positives)"
    )

n_before = len(records)
records = [r for r, keep in zip(records, keep_mask) if keep]
n_after = len(records)
print(f"Total annotated after dedup: {n_after} images")
print(f"   Duplicates removed: {n_before - n_after}")


Datasets ready
   chitholian: 665 images (1740 gt boxes)
   andrewmvd: 665 images (1740 gt boxes)
   ashishkumar: 674 images (1371 gt boxes)

Total annotated before dedup: 2004 images
Computing normalized pixel arrays for dedup ...

Removed-pair diff stats: min=0.000 median=0.140 max=0.997
   (all removed pairs should sit well below DEDUP_THRESHOLD=1.0 -- if max is close to the threshold, some may be false positives)
Total annotated after dedup: 926 images
   Duplicates removed: 1078


Build YOLO Dataset (train/val split)

In [5]:
# Shuffles the deduplicated records under seed 42, splits 80/20, and writes the YOLO-format image and label directories plus data.yaml.
import random, yaml

random.seed(42)
random.shuffle(records)
split_idx = int(len(records) * 0.8)
train_recs, val_recs = records[:split_idx], records[split_idx:]
print(f"Train: {len(train_recs)} | Val: {len(val_recs)}")

YOLO_DIR = os.path.join(ROOT, "yolo_dataset")
DATA_YAML = f"{YOLO_DIR}/data.yaml"

for split in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{YOLO_DIR}/{split}", exist_ok=True)


def convert_to_yolo(rec_list, split):
    """Write YOLO-format label files and copy images."""
    written = 0
    for rec in rec_list:
        img = cv2.imread(str(rec["image_path"]))
        if img is None:
            continue
        h, w = img.shape[:2]
        dst_img = f"{YOLO_DIR}/images/{split}/{rec['image_path'].name}"
        shutil.copy(str(rec["image_path"]), dst_img)
        lbl_path = f"{YOLO_DIR}/labels/{split}/{rec['image_path'].stem}.txt"
        with open(lbl_path, "w") as f:
            for box in rec["gt_boxes"]:
                x1, y1, x2, y2 = box
                cx = ((x1 + x2) / 2) / w
                cy = ((y1 + y2) / 2) / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h
                f.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
        written += 1
    return written


n_train = convert_to_yolo(train_recs, "train")
n_val = convert_to_yolo(val_recs, "val")

data_cfg = {
    "path": YOLO_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["pothole"],
}
with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f)

print(f"YOLO dataset ready -- train:{n_train}, val:{n_val}")
print(f"   YAML: {DATA_YAML}")

Train: 740 | Val: 186
YOLO dataset ready -- train:740, val:186
   YAML: ./yolo_dataset/data.yaml
